In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray as rxr
import cartopy.crs as ccrs

In [2]:
x_data = xr.open_dataset("../../data/CanESM_1850-2100_tas.nc", engine="netcdf4")
y_data_rsut = xr.open_dataset("../../data/CanESM_1850-2100_rsutcre.nc", engine="netcdf4")
y_data_rlut = xr.open_dataset("../../data/CanESM_1850-2100_rlutcre.nc", engine="netcdf4")

print("Input dataset (SST): CanESM_1850-2100_tas.nc")
print("Label dataset (CRE): CanESM_1850-2100_rsutcre.nc")
print("Label dataset (CRE): CanESM_1850-2100_rlutcre.nc")

Input dataset (SST): CanESM_1850-2100_tas.nc
Label dataset (CRE): CanESM_1850-2100_rsutcre.nc
Label dataset (CRE): CanESM_1850-2100_rlutcre.nc


In [3]:
print("Input dataset (x_data / SST):")
print(x_data)
print("\nLabel dataset (y_data_rsut / RSUT):")
print(y_data_rsut)
print("\nLabel dataset (y_data_rlut / RLUT):")
print(y_data_rlut)

def select_var(ds, preferred_names):
    for name in preferred_names:
        if name in ds.data_vars:
            return name
    non_bnds = [name for name in ds.data_vars if not name.endswith("_bnds")]
    if not non_bnds:
        raise ValueError("No usable data variable found in dataset")
    return non_bnds[0]

x_var = select_var(x_data, ["tas"])
y_var_rsut = select_var(y_data_rsut, ["cre"])
y_var_rlut = select_var(y_data_rlut, ["cre"])


x_da = x_data[x_var]
y_da_rsut = y_data_rsut[y_var_rsut]
y_da_rlut = y_data_rlut[y_var_rlut]

print("\nSelected input variable:", x_var)
print(x_da)
print("\nSelected label variable (RSUT):", y_var_rsut)
print(y_da_rsut)
print("\nSelected label variable (RLUT):", y_var_rlut)
print(y_da_rlut)

Input dataset (x_data / SST):
<xarray.Dataset> Size: 2GB
Dimensions:  (member: 25, time: 3012, lat: 64, lon: 128)
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
    height   float64 8B ...
Data variables:
    tas      (member, time, lat, lon) float32 2GB ...

Label dataset (y_data_rsut / RSUT):
<xarray.Dataset> Size: 2GB
Dimensions:  (member: 25, time: 3012, lat: 64, lon: 128)
Coordinates:
  * member   (member) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * time     (time) object 24kB 1850-01-16 12:00:00 ... 2100-12-16 12:00:00
  * lat      (lat) float64 512B -87.86 -85.1 -82.31 -79.53 ... 82.31 85.1 87.86
  * lon      (lon) float64 1kB 0.0 2.812 5.625 8.438 ... 348.8 351.6 354.4 357.2
Data vari

In [4]:
print("x_da dimensions:", x_da.dims)
print("x_da coordinates:", list(x_da.coords))
print("x_da attributes keys:", list(x_da.attrs.keys()))

print("\ny_da dimensions:", y_da_rsut.dims)
print("y_da coordinates:", list(y_da_rsut.coords))
print("y_da attributes keys:", list(y_da_rsut.attrs.keys()))

print("\ny_da dimensions:", y_da_rlut.dims)
print("y_da coordinates:", list(y_da_rlut.coords))
print("y_da attributes keys:", list(y_da_rlut.attrs.keys()))

x_da dimensions: ('member', 'time', 'lat', 'lon')
x_da coordinates: ['time', 'lat', 'lon', 'height', 'member']
x_da attributes keys: ['standard_name', 'long_name', 'comment', 'units', 'original_name', 'history', 'cell_methods', 'cell_measures']

y_da dimensions: ('member', 'time', 'lat', 'lon')
y_da coordinates: ['time', 'lat', 'lon', 'member']
y_da attributes keys: ['units', 'cell_methods', 'cell_measures', 'history', 'long_name', 'description']

y_da dimensions: ('member', 'time', 'lat', 'lon')
y_da coordinates: ['time', 'lat', 'lon', 'member']
y_da attributes keys: ['units', 'cell_methods', 'cell_measures', 'long_name', 'description']


In [5]:
x_da, y_da_rsut, y_da_rlut = xr.align(x_da, y_da_rsut, y_da_rlut, join="inner")

if "member" not in x_da.dims or "member" not in y_da_rsut.dims or "member" not in y_da_rlut.dims:
    raise ValueError("Expected a 'member' dimension in input and both label datasets.")

n_members = x_da.sizes["member"]
if n_members != 25:
    raise ValueError(f"Expected 25 members, found {n_members}.")

rng = np.random.default_rng(42)
member_idx = rng.permutation(n_members)

train_idx = member_idx[:17]
val_idx = member_idx[17:21]
test_idx = member_idx[21:]

X_train = x_da.isel(member=train_idx)
X_val = x_da.isel(member=val_idx)
X_test = x_da.isel(member=test_idx)

y_train_rsut = y_da_rsut.isel(member=train_idx)
y_val_rsut = y_da_rsut.isel(member=val_idx)
y_test_rsut = y_da_rsut.isel(member=test_idx)

y_train_rlut = y_da_rlut.isel(member=train_idx)
y_val_rlut = y_da_rlut.isel(member=val_idx)
y_test_rlut = y_da_rlut.isel(member=test_idx)

print("Random member split (train/val/test):", len(train_idx), len(val_idx), len(test_idx))
print("Train members:", train_idx)
print("Validation members:", val_idx)
print("Test members:", test_idx)

print("X (SST) member sizes:", X_train.sizes["member"], X_val.sizes["member"], X_test.sizes["member"])
print("y (RSUT) member sizes:", y_train_rsut.sizes["member"], y_val_rsut.sizes["member"], y_test_rsut.sizes["member"])
print("y (RLUT) member sizes:", y_train_rlut.sizes["member"], y_val_rlut.sizes["member"], y_test_rlut.sizes["member"])

Random member split (train/val/test): 17 4 4
Train members: [15 16 19 20  9  7 17 24  6 10  3  0 21 18 12  5 11]
Validation members: [14 23  2  4]
Test members: [22  1 13  8]
X (SST) member sizes: 17 4 4
y (RSUT) member sizes: 17 4 4
y (RLUT) member sizes: 17 4 4
